# V016: fixed postprocessing screen on saved A predictions

Discussion-inspired (source/provenance in `research/DISCUSSION_INSPIRED_ROADMAP_v015.md`). Three policies were frozen before scoring: nearest training-only pressure codebook; weighted median midpoint with existing fixed weights; their combination. No new model fitting, arbitrary weight search or official-score feedback.

Reference is the V013 A-selected six-member curve-loss mixture, **provisional until the existing V014 confirmation finishes**. This screen does not alter that confirmation and cannot rescue a failed primary E result. Any eligible postprocessing recipe requires separate future grouped confirmation, with its codebook refitted on that split's fit labels only. Median model predictions and median absolute error are distinct quantities. Native mean inspiration-only MAE remains the objective.


In [1]:
from pathlib import Path
import json,hashlib,time
import numpy as np
import pandas as pd
root=Path.cwd();out=root/"outputs/postprocessing_screen_v016";out.mkdir(parents=True,exist_ok=True)
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
planpath=root/"research/postprocessing_protocol_v016.json"
assert sha(planpath)=="e91b66f4d7047dc0991f7735955d3f3a594073806bc443e206720f641b73b040"
plan=json.loads(planpath.read_text());started=time.monotonic()
prior=json.loads((root/"research/protocol_v013.json").read_text());screen=json.loads((root/"outputs/controlled_experiments_v013/screen_summary.json").read_text())
assert sha(root/"data/train.csv")=="2dc30dc5c8e603c8c68200830eab7ed8c7af42c823f8ae68b0beb663eea817d7"
df=pd.read_csv(root/"data/train.csv",usecols=["id","breath_id","u_out","pressure"],dtype={"id":"int64","breath_id":"int32","u_out":"int8","pressure":"float32"})
bids=df.breath_id.to_numpy()[::80];pressure=df.pressure.to_numpy().reshape(-1,80);phase=df.u_out.to_numpy().reshape(-1,80)
with np.load(root/"outputs/improvements_v005/sequence/split_breath_ids.npz") as z:
    fit=np.sort(pd.Index(bids).get_indexer(z["A_fit"]));val=np.sort(pd.Index(bids).get_indexer(z["A_validation"]))
assert len(fit)==61114 and len(val)==6791 and not np.intersect1d(fit,val).size
mask=phase[val]==0;counts=mask.sum(1);truth=pressure[val][mask].astype(np.float64)
assert np.array_equal(df.id.to_numpy().reshape(-1,80)[val][mask],np.load(root/"outputs/controlled_experiments_v013/A_row_ids.npy"))
codebook=np.unique(pressure[fit][phase[fit]==0]).astype(np.float64)
assert np.isfinite(codebook).all() and np.all(np.diff(codebook)>0)
np.save(out/"A_fit_pressure_codebook.npy",codebook)
predictions={}
for name,path in prior["reference_prediction_files"].items():
    assert sha(root/path)==prior["inputs"][path]
    predictions[name]=np.load(root/path)
a=screen["records"]["deep_curve"]
assert sha(root/a["prediction_file"])==a["prediction_file_sha256"]
predictions["deep_curve"]=np.load(root/a["prediction_file"])
reference=np.load(root/plan["reference_array"])
assert sha(root/plan["reference_array"])==plan["reference_sha256"]
assert np.array_equal(reference,sum(w*predictions[k] for k,w in plan["weights"].items()))
assert abs(np.abs(reference-truth).mean()-plan["reference_MAE"])<1e-15

def snap(p):
    right=np.searchsorted(codebook,p).clip(0,len(codebook)-1)
    left=np.maximum(right-1,0)
    choose_left=np.abs(p-codebook[left])<=np.abs(p-codebook[right])
    return np.where(choose_left,codebook[left],codebook[right])
values=np.stack([predictions[k] for k in plan["weights"]])
weights=np.asarray(list(plan["weights"].values()))
order=np.argsort(values,axis=0,kind="stable")
sorted_values=np.take_along_axis(values,order,axis=0)
sorted_weights=np.take_along_axis(np.broadcast_to(weights[:,None],values.shape),order,axis=0)
cumulative=sorted_weights.cumsum(0)
lo=(cumulative>=.5).argmax(0);hi=(cumulative>.5).argmax(0);cols=np.arange(len(truth))
median=.5*(sorted_values[lo,cols]+sorted_values[hi,cols])
candidates={"nearest_training_pressure_codebook":snap(reference),"weighted_median_midpoint":median,"weighted_median_midpoint_then_nearest_training_pressure_codebook":snap(median)}
assert set(candidates)==set(plan["candidate_policies"])
old=np.abs(reference-truth);results=[]
for name,pred in candidates.items():
    assert pred.dtype==np.float64 and pred.shape==truth.shape and np.isfinite(pred).all()
    error=np.abs(pred-truth);gain=float(old.mean()-error.mean())
    sums=np.add.reduceat(error-old,np.r_[0,np.cumsum(counts)[:-1]]);rng=np.random.default_rng(plan["bootstrap_seed"]);draws=[]
    for _ in range(10):
        take=rng.integers(0,len(counts),size=(100,len(counts)));draws.extend(sums[take].sum(1)/counts[take].sum(1))
    low,high=np.quantile(draws,[.025,.975]);path=out/(name+"_A_predictions.npy");np.save(path,pred)
    results.append(dict(name=name,MAE=float(error.mean()),median_absolute_error=float(np.median(error)),gain=gain,bootstrap_95=[float(low),float(high)],eligible=bool(gain>=.003 and high<0),prediction_file=str(path.relative_to(root)),prediction_sha256=sha(path)))
report=dict(status="complete",protocol_sha256=sha(planpath),reference_A_MAE=float(old.mean()),reference_is_provisional_until_E=True,results=results,training_only_codebook_levels=len(codebook),codebook_sha256=sha(out/"A_fit_pressure_codebook.npy"),validation_breaths=6791,validation_scored_points=len(truth),model_fits=0,current_E_unchanged=True,no_E_result_read=True,official_best_preserved=.18025,submission_created=False,runtime_seconds=time.monotonic()-started)
(out/"results.json").write_text(json.dumps(report,indent=2)+"\n")
print("Provisional reference A MAE:",report["reference_A_MAE"])
print(pd.DataFrame([{k:r[k] for k in ["name","MAE","gain","eligible"]} for r in results]).to_string(index=False))
print("No E results read and no submission created.")


Provisional reference A MAE: 0.1862642374201326
                                                            name      MAE     gain  eligible
                              nearest_training_pressure_codebook 0.184947 0.001318     False
                                        weighted_median_midpoint 0.184277 0.001988     False
weighted_median_midpoint_then_nearest_training_pressure_codebook 0.182922 0.003342      True
No E results read and no submission created.
